# Notebook 10 Part 2: Cache and Persist in PySpark
 PySpark Cache and Persist Explained Speed Up Repeated Operations


## 1. Why Do We Need Cache?

Spark is lazy. It does not immediately run transformations.

Example:

```python
df_filtered = df.filter(df.amount > 30000)
```

Spark only builds the plan. When an action runs, Spark executes the plan.

If the same transformed DataFrame is used many times, Spark may compute the same transformation again and again.

`cache()` helps Spark store the computed result and reuse it.

## 2. Simple Analogy

Imagine you prepared a report after many calculations.

If you need the report again and again:

```text
Without cache → calculate again every time
With cache    → save once and reuse
```

In Spark:

```text
cache() = store this computed DataFrame for reuse
```

## 3. Basic Syntax

```python
df_cached = df.cache()
df_cached.count()
```

Important:

```text
cache() is lazy.
It stores data only after an action runs.
```

`count()` is commonly used to materialize the cache.

## 4. Windows Local Setup

In [1]:
import os
import sys

os.environ["PYSPARK_PYTHON"] = sys.executable
os.environ["PYSPARK_DRIVER_PYTHON"] = sys.executable
os.environ["SPARK_LOCAL_IP"] = "127.0.0.1"
os.environ["SPARK_LOCAL_HOSTNAME"] = "localhost"

## 5. Create SparkSession

In [2]:
from pyspark.sql import SparkSession

spark = SparkSession.builder \
    .appName("Cache_and_Persist_in_PySpark") \
    .master("local[1]") \
    .config("spark.python.worker.reuse", "false") \
    .config("spark.driver.host", "127.0.0.1") \
    .config("spark.driver.bindAddress", "127.0.0.1") \
    .getOrCreate()

spark.sparkContext.setLogLevel("ERROR")

## 6. Create Sample DataFrame

In [4]:
customers_data = [
    (1, "Amit", "Delhi",490490),
    (2, "Priya", "Mumbai", 12021092),
    (3, "Rahul", "Pune", 8213),
    (4, "Sneha", "Delhi",23412),
    (5, "Karan", "Jaipur",231333)
]

customers_df = spark.createDataFrame(
    customers_data,
    ["customer_id", "customer_name", "city", "amount"]
)

customers_df.show()

+-----------+-------------+------+--------+
|customer_id|customer_name|  city|  amount|
+-----------+-------------+------+--------+
|          1|         Amit| Delhi|  490490|
|          2|        Priya|Mumbai|12021092|
|          3|        Rahul|  Pune|    8213|
|          4|        Sneha| Delhi|   23412|
|          5|        Karan|Jaipur|  231333|
+-----------+-------------+------+--------+



## 7. Create a Transformed DataFrame

These are transformations:

```text
filter()
withColumn()
```

No action has run yet.

In [5]:
from pyspark.sql.functions import col, when

df_high_value = customers_df.filter( col("amount") > 70000).withColumn( "sales_category", 
                                                                           when(col("amount") >=100000, "Very High")
                                                                               .otherwise("high"))

## 8. Use Same DataFrame Multiple Times Without Cache

Each action may trigger computation again.

In [6]:
df_high_value.count()

3

In [7]:
df_high_value.groupBy("city").count().show()

+------+-----+
|  city|count|
+------+-----+
|Mumbai|    1|
| Delhi|    1|
|Jaipur|    1|
+------+-----+



## 9. Apply `cache()`

This marks the DataFrame to be cached.

In [8]:
df_high_value_cached = df_high_value.cache()

print("Cache instruction added.")

Cache instruction added.


Important:

```text
cache() only gives an instruction.
Data is not stored yet.
```

## 10. Materialize the Cache

Now run an action.

In [9]:
df_high_value_cached.count()

3

After this action, Spark can reuse cached data.

## 11. Reuse Cached DataFrame

In [10]:
df_high_value_cached.groupBy("city").count().show()

+------+-----+
|  city|count|
+------+-----+
|Mumbai|    1|
| Delhi|    1|
|Jaipur|    1|
+------+-----+



## 12. Check Storage Level

In [11]:
df_high_value_cached.storageLevel

StorageLevel(True, True, False, True, 1)

| Position | Value   | Meaning        | Simple Meaning                        |
| -------- | ------- | -------------- | ------------------------------------- |
| 1        | `True`  | `useDisk`      | Spark can store data on disk          |
| 2        | `True`  | `useMemory`    | Spark can store data in memory        |
| 3        | `False` | `useOffHeap`   | Spark will not use off-heap memory    |
| 4        | `True`  | `deserialized` | Data is stored in ready-to-use format |
| 5        | `1`     | `replication`  | Only 1 copy of cached data            |


The output tells how the DataFrame is stored.

For DataFrames, `cache()` usually uses a memory-and-disk style storage level.

## 13. Remove Cache Using `unpersist()`

In [12]:
df_high_value_cached.unpersist()

print("Cache removed.")

Cache removed.


Use `unpersist()` when cached data is no longer needed.

Why?

```text
Cached data uses memory.
Unnecessary cache can create memory pressure.
```

# 14. What Is `persist()`?

`persist()` is similar to `cache()`, but it gives more control.

Examples:

```python
df.persist(StorageLevel.MEMORY_ONLY)
df.persist(StorageLevel.MEMORY_AND_DISK)
df.persist(StorageLevel.DISK_ONLY)
```

Simple meaning:

```text
cache()   = simple default storage
persist() = choose storage level
```

## 15. Import StorageLevel

In [13]:
from pyspark import StorageLevel

## 16. Persist with MEMORY_ONLY

In [ ]:
df_memory_only = df_high_value.persist(StorageLevel.MEMORY_ONLY)

df_memory_only.count()

print(df_memory_only.storageLevel)

df_memory_only.unpersist()

`MEMORY_ONLY` means Spark tries to keep the DataFrame only in memory.

If memory is not enough, some partitions may not be cached.

## 17. Persist with MEMORY_AND_DISK

In [ ]:
df_memory_disk = df_high_value.persist(StorageLevel.MEMORY_AND_DISK)

df_memory_disk.count()

print(df_memory_disk.storageLevel)

df_memory_disk.unpersist()

`MEMORY_AND_DISK` means:

```text
Store in memory if possible.
If memory is not enough, spill to disk.
```

This is safer for large data.

## 18. Common Storage Levels

| Storage level | Meaning |
|---|---|
| `MEMORY_ONLY` | Store only in memory |
| `MEMORY_AND_DISK` | Store in memory, spill to disk if needed |
| `DISK_ONLY` | Store only on disk |
| `MEMORY_ONLY_SER` | Store serialized data in memory |
| `MEMORY_AND_DISK_SER` | Store serialized data in memory and disk |

For beginners, focus mainly on:

```text
MEMORY_ONLY
MEMORY_AND_DISK
DISK_ONLY
```

## 19. cache() vs persist()

| Point | cache() | persist() |
|---|---|---|
| Purpose | Reuse computed DataFrame | Reuse computed DataFrame |
| Control | Less control | More control |
| Storage level | Default storage level | User can choose storage level |
| Beginner usage | Very common | Used when storage choice matters |
| Example | `df.cache()` | `df.persist(StorageLevel.MEMORY_AND_DISK)` |

Simple line:

```text
cache() is a shortcut.
persist() is configurable caching.
```

#  Cache with Spark SQL Temporary View

We can also cache a temporary view/table.

In [15]:
df.createOrReplaceTempView("sales")
                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                                            
spark.catalog.cacheTable("sales")

print("Is sales cached?", spark.catalog.isCached("sales"))

Is sales cached? True


In [ ]:
spark.sql("SELECT city, SUM(amount) AS total_amount FROM sales GROUP BY city").show()